# Plan B — hOCR Chunked Reading + Combined Segmentation & Extraction

Uses the hOCR output (HTML with embedded word coordinates) to reconstruct reading order, then feeds document chunks to a text LLM via Ollama that handles **both segmentation and extraction in one pass**. Simpler pipeline, trades spatial precision for flexibility.

**Full pipeline:**
1. Install system dependencies (Tesseract, aspell, poppler-utils)
2. Convert PDF to page PNGs via `pdftoppm`
3. Run Tesseract OCR → `.txt` (plain text) + `.hocr` (HTML with bounding boxes)
4. OCR quality gate (aspell word validity check)
5. Reconstruct reading order from hOCR bounding boxes
6. Chunk the document (~800 tokens, 100-token overlap)
7. Combined segmentation + extraction via `ollama run`
8. Deduplicate by `(title, catalog_number)`
9. Local entity resolution
10. Write QuickStatements-compatible CSV + human-readable CSV

In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────
# Set these before running. All other paths are derived from pdf_file + output_dir.

pdf_file       = "baldessari-sprengel.pdf"   # input PDF
tesseract_lang = "eng+deu"                   # Tesseract language(s)
pdf_dpi        = 300                         # resolution for PDF → PNG conversion
output_dir     = "."                         # directory for all outputs
ollama_model   = "qwen2.5:14b"              # Ollama model for extraction
chunk_size     = 800                         # target words per chunk
overlap        = 100                         # overlap words between chunks

## Install Dependencies

Installs `tesseract-ocr`, `poppler-utils` (for `pdftoppm`), and `aspell` via apt if not already present. On HF Spaces these can also be listed in `packages.txt`.

In [ ]:
import shutil
import subprocess
import sys

def apt_install(*packages):
    subprocess.run(
        ["apt-get", "install", "-y", "--no-install-recommends"] + list(packages),
        check=True,
    )

checks = [
    ("pdftoppm",  ["poppler-utils"]),
    ("tesseract", ["tesseract-ocr", "tesseract-ocr-eng", "tesseract-ocr-deu"]),
    ("aspell",    ["aspell", "aspell-en", "aspell-de"]),
]
missing_pkgs = []
for binary, pkgs in checks:
    if shutil.which(binary):
        print(f"  {binary}: OK")
    else:
        print(f"  {binary}: missing — will install {pkgs}")
        missing_pkgs.extend(pkgs)

if missing_pkgs:
    print(f"\nInstalling: {' '.join(missing_pkgs)}")
    apt_install(*missing_pkgs)
    print("Installation complete.")

# Python dependencies
try:
    from bs4 import BeautifulSoup
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "beautifulsoup4", "lxml"], check=True)
    from bs4 import BeautifulSoup

try:
    import lxml  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "lxml"], check=True)

for cmd in ["tesseract --version", "pdftoppm -v", "aspell --version"]:
    r = subprocess.run(cmd.split(), capture_output=True, text=True)
    first_line = (r.stdout or r.stderr).splitlines()[0] if (r.stdout or r.stderr) else ""
    print(first_line)

In [ ]:
import re
import json
import csv
import difflib
import logging
from pathlib import Path
from bs4 import BeautifulSoup

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger(__name__)

# Derive all paths from parameters
pdf_path  = Path(pdf_file)
stem      = pdf_path.stem
_out      = Path(output_dir)
_out.mkdir(parents=True, exist_ok=True)

PAGES_DIR = _out / f"{stem}_pages"
FILELIST  = PAGES_DIR / "filelist.txt"
TXT_FILE  = _out / f"{stem}.txt"
HOCR_FILE = _out / f"{stem}.hocr"
OUT_CSV   = _out / "output-plan-b.csv"
OUT_READ  = _out / "output-plan-b-readable.csv"
FAIL_LOG  = _out / "output-plan-b-failures.txt"

print(f"PDF        : {pdf_path}  (exists: {pdf_path.exists()})")
print(f"Pages dir  : {PAGES_DIR}")
print(f"OCR text   : {TXT_FILE}")
print(f"hOCR       : {HOCR_FILE}")
print(f"Output dir : {_out.resolve()}")

# ── Entity resolution lookup tables ─────────────────────────────────────────
CREATOR_QIDS = {
    "John Baldessari": "Q312793",
    "Baldessari": "Q312793",
}
OBJECT_TYPE_QIDS = {
    "painting": "Q3305213",
    "photograph": "Q125191",
    "gelatin silver print": "Q3286805",
    "video": "Q34508",
    "drawing": "Q93184",
    "print": "Q11060274",
    "collage": "Q170571",
    "mixed media": "Q1902763",
}
MATERIAL_QIDS = {
    "acrylic": "Q207849",
    "oil": "Q296955",
    "canvas": "Q4259259",
    "gelatin silver": "Q3286805",
    "photograph": "Q125191",
    "ink": "Q127418",
    "pencil": "Q14674",
    "paper": "Q11472",
}

## PDF → Page Images

Converts each PDF page to a PNG at the specified DPI using `pdftoppm`. Skip this cell if the images already exist.

In [ ]:
PAGES_DIR.mkdir(exist_ok=True)

existing = sorted(PAGES_DIR.glob("page-*.png"))
if existing:
    print(f"Found {len(existing)} existing page images — skipping conversion.")
    print(f"  Delete {PAGES_DIR} to re-run from scratch.")
else:
    print(f"Converting {pdf_path.name} to PNG at {pdf_dpi} DPI...")
    result = subprocess.run(
        ["pdftoppm", "-r", str(pdf_dpi), "-png", str(pdf_path), str(PAGES_DIR / "page")],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"pdftoppm stderr: {result.stderr}")
        raise RuntimeError("pdftoppm failed")
    existing = sorted(PAGES_DIR.glob("page-*.png"))
    print(f"Created {len(existing)} page images.")

page_numbers = sorted(
    int(re.search(r"page-(\d+)\.png", p.name).group(1))
    for p in existing
)
print(f"Pages: {page_numbers[0]:02d} to {page_numbers[-1]:02d}")

## Run Tesseract OCR

Builds a filelist from the page PNGs and runs Tesseract once to produce both plain text (`.txt`) and hOCR (`.hocr`). Skip this cell if the OCR output files already exist.

In [ ]:
if TXT_FILE.exists() and HOCR_FILE.exists():
    print(f"OCR output already exists — skipping Tesseract.")
    print(f"  {TXT_FILE.name}: {TXT_FILE.stat().st_size:,} bytes")
    print(f"  {HOCR_FILE.name}: {HOCR_FILE.stat().st_size:,} bytes")
    print(f"  Delete these files to re-run OCR.")
else:
    FILELIST.write_text(
        "\n".join(str(PAGES_DIR / f"page-{n:02d}.png") for n in page_numbers),
        encoding="utf-8",
    )
    print(f"Filelist written: {len(page_numbers)} pages")

    output_base = str(_out / stem)
    print(f"Running: tesseract {FILELIST.name} {stem} -l {tesseract_lang} txt hocr")
    result = subprocess.run(
        ["tesseract", str(FILELIST), output_base, "-l", tesseract_lang, "txt", "hocr"],
        capture_output=True, text=True,
    )
    if result.stderr:
        print(f"Tesseract stderr:\n{result.stderr[:500]}")
    if result.returncode != 0 and not TXT_FILE.exists():
        raise RuntimeError("Tesseract failed — see stderr above")

    print()
    for f in [TXT_FILE, HOCR_FILE]:
        if f.exists():
            print(f"  {f.name}: {f.stat().st_size:,} bytes")
        else:
            print(f"  {f.name}: NOT CREATED")

## Step 0 — OCR Quality Gate

Score the Tesseract output using aspell (English + German union). Threshold `>= 0.80` for Plan B — the LLM heals minor OCR errors in context, so a lower threshold is acceptable.

In [ ]:
def step0_quality_gate():
    print("=== Step 0: OCR Quality Gate ===")
    text = TXT_FILE.read_text(encoding="utf-8", errors="replace")
    words = re.findall(r"[A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df'-]+", text)
    total_words = len(words)

    if total_words == 0:
        print("WARN: no words found in txt file")
        return 0.0, 0

    word_input = "\n".join(words).encode("utf-8")
    unknown_en: set = set()
    unknown_de: set = set()

    for lang, container in [("en", unknown_en), ("de", unknown_de)]:
        try:
            result = subprocess.run(
                ["aspell", "list", f"--lang={lang}"],
                input=word_input,
                capture_output=True,
                timeout=120,
            )
            if result.returncode == 0:
                for w in result.stdout.decode("utf-8", errors="replace").splitlines():
                    w = w.strip()
                    if w:
                        container.add(w)
        except (subprocess.TimeoutExpired, FileNotFoundError) as e:
            log.warning("aspell %s failed: %s", lang, e)

    unknown_words  = unknown_en & unknown_de
    validity_ratio = (total_words - len(unknown_words)) / total_words

    print(f"Total words:    {total_words:,}")
    print(f"Unknown (en):   {len(unknown_en):,}")
    print(f"Unknown (de):   {len(unknown_de):,}")
    print(f"Unknown (both): {len(unknown_words):,}")
    print(f"Validity ratio: {validity_ratio:.4f}")
    gate = "PASS" if validity_ratio >= 0.80 else "FAIL"
    print(f"OCR Quality Gate: {gate}  (threshold 0.80 for Plan B)")
    return validity_ratio, total_words

In [ ]:
validity_ratio, total_words = step0_quality_gate()

## Step 1 — Reconstruct Reading Order from hOCR

Parse hOCR with BeautifulSoup, sort lines by y-coordinate per page, and insert paragraph breaks where the vertical gap between lines exceeds 20px.

In [ ]:
def parse_bbox(title_str):
    m = re.search(r"bbox\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)", title_str or "")
    return tuple(int(m.group(i)) for i in range(1, 5)) if m else None


def step1_reconstruct_hocr():
    print("=== Step 1: Reconstruct reading order from hOCR ===")
    raw  = HOCR_FILE.read_bytes()
    soup = BeautifulSoup(raw, "lxml")

    pages = soup.find_all("div", class_="ocr_page")
    print(f"Found {len(pages)} pages")

    all_chunks = []
    for page_idx, page in enumerate(pages, start=1):
        page_lines = []
        for line_span in page.find_all("span", class_="ocr_line"):
            bbox = parse_bbox(line_span.get("title", ""))
            if bbox is None:
                continue
            words = [
                w.get_text(" ", strip=True)
                for w in line_span.find_all("span", class_="ocrx_word")
                if w.get_text(strip=True)
            ]
            line_text = " ".join(words).strip()
            if line_text:
                page_lines.append((bbox[1], bbox[3], line_text))

        page_lines.sort(key=lambda x: x[0])

        parts = [f"--- PAGE {page_idx} ---"]
        prev_y1 = None
        for y0, y1, line_text in page_lines:
            if prev_y1 is not None and y0 - prev_y1 > 20:
                parts.append("")
            parts.append(line_text)
            prev_y1 = y1
        all_chunks.append("\n".join(parts))

    full_text = "\n\n".join(all_chunks)
    print(f"Reconstructed text length: {len(full_text):,} chars")
    return full_text

In [ ]:
full_text = step1_reconstruct_hocr()

## Step 2 — Chunk the Document

Split into overlapping word-count chunks so entries that straddle paragraph breaks are not lost.

In [ ]:
def step2_chunk(full_text, chunk_size=chunk_size, overlap=overlap):
    print("=== Step 2: Chunking document ===")
    paragraphs = re.split(r"\n{2,}", full_text)
    chunks = []
    current_words = []
    current_count = 0

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
        para_words = para.split()
        if current_count + len(para_words) > chunk_size and current_count > 0:
            chunks.append(" ".join(current_words).strip())
            current_words = current_words[-overlap:] + para_words
        else:
            current_words.extend(para_words)
        current_count = len(current_words)

    if current_words:
        chunks.append(" ".join(current_words).strip())

    print(f"Total chunks: {len(chunks)}")
    return chunks

In [ ]:
chunks     = step2_chunk(full_text)
num_chunks = len(chunks)

## Step 3 — Combined Segmentation + Extraction via Ollama

Each chunk is sent to the LLM with a single prompt that both finds catalog entries and extracts structured fields. Returns a JSON array per chunk.

In [ ]:
PROMPT = (
    "You are processing text from an exhibition catalog for John Baldessari at "
    "Sprengel Museum Hannover. Find any artwork catalog entries in this text — "
    "these are structured entries listing an artwork with its title, date, "
    "materials, and/or dimensions. Ignore essay prose and biographical text. "
    "For each catalog entry found, extract the fields listed below. Return ONLY "
    "a JSON array of objects (one per artwork). If no catalog entries are found, "
    "return an empty array []. Fields for each object: title (string), creator "
    "(string, default 'John Baldessari'), date (string, year or range), "
    "object_type (string), materials (array of strings), height_cm (number or null), "
    "width_cm (number or null), depth_cm (number or null), inventory_number "
    "(string or null), catalog_number (string or null). Do NOT invent values. "
    "Use null for missing fields."
)


def extract_json_array(text):
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, list): return obj
    except json.JSONDecodeError:
        pass
    start = text.find("[")
    if start != -1:
        depth = 0
        for idx in range(start, len(text)):
            if text[idx] == "[": depth += 1
            elif text[idx] == "]":
                depth -= 1
                if depth == 0:
                    try:
                        obj = json.loads(text[start : idx + 1])
                        if isinstance(obj, list): return obj
                    except json.JSONDecodeError:
                        pass
                    break
    return None


def run_ollama(chunk_text, chunk_idx, total):
    print(f"  Chunk {chunk_idx}/{total} ({len(chunk_text.split())} words)...", end=" ", flush=True)
    try:
        result = subprocess.run(
            ["ollama", "run", ollama_model, PROMPT],
            input=chunk_text.encode("utf-8"),
            capture_output=True,
            timeout=300,
        )
        output = result.stdout.decode("utf-8", errors="replace").strip()
        stderr = result.stderr.decode("utf-8", errors="replace").strip()
    except subprocess.TimeoutExpired:
        print("TIMEOUT")
        with open(FAIL_LOG, "a") as f:
            f.write(f"Chunk {chunk_idx}: timeout\n")
        return []
    except Exception as e:
        print(f"ERROR: {e}")
        with open(FAIL_LOG, "a") as f:
            f.write(f"Chunk {chunk_idx}: {e}\n")
        return []

    records = extract_json_array(output)
    if records is None:
        print(f"PARSE FAIL (len={len(output)})")
        with open(FAIL_LOG, "a") as f:
            f.write(f"=== Chunk {chunk_idx} parse failure ===\nSTDERR: {stderr}\nOUTPUT:\n{output}\n\n")
        return []

    records = [r for r in records if isinstance(r, dict)]
    print(f"found {len(records)} records")
    return records


def step3_extract(chunks):
    print("=== Step 3: Extraction via Ollama ===")
    FAIL_LOG.write_text("", encoding="utf-8")
    all_records = []
    for idx, chunk in enumerate(chunks, start=1):
        all_records.extend(run_ollama(chunk, idx, len(chunks)))
    print(f"Raw records found: {len(all_records)}")
    return all_records

In [ ]:
raw_records = step3_extract(chunks)
raw_count   = len(raw_records)

## Step 4 — Deduplicate

Entries appearing in multiple chunks (due to overlap) are merged, preferring the record with more non-null fields.

In [ ]:
def non_null_count(rec):
    return sum(1 for v in rec.values() if v is not None and v != "" and v != [])


def step4_deduplicate(records):
    print("=== Step 4: Deduplication ===")
    seen = {}
    for rec in records:
        title = (rec.get("title") or "").strip().lower()
        cat   = (rec.get("catalog_number") or "").strip().lower()
        key   = (title, cat)
        if key not in seen or non_null_count(rec) > non_null_count(seen[key]):
            seen[key] = rec

    by_title = {}
    for rec in seen.values():
        title = (rec.get("title") or "").strip().lower()
        cat   = rec.get("catalog_number")
        if cat or title not in by_title or non_null_count(rec) > non_null_count(by_title[title]):
            by_title[title] = rec

    result = list(by_title.values())
    print(f"After dedup: {len(result)} records (from {len(records)} raw)")
    return result

In [ ]:
deduped = step4_deduplicate(raw_records)

## Step 5 — Entity Resolution

Fuzzy-match creator names, material terms, and object types to Wikidata Q-IDs using `difflib`. No external APIs.

In [ ]:
def resolve_qid(value, lookup_dict):
    if not value: return None
    if value in lookup_dict: return lookup_dict[value]
    val_lower = value.lower()
    for k, v in lookup_dict.items():
        if k.lower() == val_lower: return v
    matches = difflib.get_close_matches(value, lookup_dict.keys(), n=1, cutoff=0.8)
    return lookup_dict[matches[0]] if matches else None


def step5_resolve(records):
    print("=== Step 5: Entity resolution ===")
    for rec in records:
        creator = rec.get("creator") or "John Baldessari"
        rec["creator"] = creator
        rec["creator_qid"] = resolve_qid(creator, CREATOR_QIDS) or "Q312793"
        rec["object_type_qid"] = resolve_qid(rec.get("object_type") or "", OBJECT_TYPE_QIDS)

        mats = rec.get("materials") or []
        if isinstance(mats, str):
            mats = [m.strip() for m in re.split(r"[,;/]", mats) if m.strip()]
        mat_qids = []
        for m in mats:
            qid = resolve_qid(m, MATERIAL_QIDS)
            if not qid:
                qid = next((v for k, v in MATERIAL_QIDS.items()
                            if k.lower() in m.lower() or m.lower() in k.lower()), None)
            mat_qids.append(qid)
        rec["materials"]     = mats
        rec["materials_qids"] = mat_qids
    return records

In [ ]:
resolved = step5_resolve(deduped)

## Step 6 — Write Outputs

Writes two CSVs:
- `output-plan-b.csv` — QuickStatements-compatible (Wikidata P-number headers)
- `output-plan-b-readable.csv` — human-readable field name headers

In [ ]:
def format_date_qs(date_str):
    if not date_str: return None
    m = re.search(r"\b(\d{4})\b", str(date_str))
    return f"+{m.group(1)}-00-00T00:00:00Z/9" if m else None


def step6_write_csv(records):
    print("=== Step 6: Writing CSV outputs ===")
    qs_cols = ["qid", "P31", "P1476", "P170", "P571", "P276", "P195",
               "P186", "P2048", "P2049", "P2610", "P217", "P528", "P1431"]

    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=qs_cols)
        writer.writeheader()
        for rec in records:
            mat_qids  = rec.get("materials_qids") or []
            first_mat = mat_qids[0] if mat_qids else None
            title     = rec.get("title") or ""
            writer.writerow({
                "qid":   "CREATE",
                "P31":   rec.get("object_type_qid") or "",
                "P1476": f'"{title}"@en' if title else "",
                "P170":  rec.get("creator_qid") or "Q312793",
                "P571":  format_date_qs(rec.get("date")) or "",
                "P276":  "Q673907",
                "P195":  "Q673907",
                "P186":  first_mat or "",
                "P2048": rec.get("height_cm") if rec.get("height_cm") is not None else "",
                "P2049": rec.get("width_cm")  if rec.get("width_cm")  is not None else "",
                "P2610": rec.get("depth_cm")  if rec.get("depth_cm")  is not None else "",
                "P217":  rec.get("inventory_number") or "",
                "P528":  rec.get("catalog_number")   or "",
                "P1431": "Q138573075",
            })
    print(f"  Wrote {OUT_CSV} ({OUT_CSV.stat().st_size:,} bytes, {len(records)} rows)")

    read_cols = ["title", "creator", "creator_qid", "date", "object_type",
                 "object_type_qid", "materials", "materials_qids",
                 "height_cm", "width_cm", "depth_cm",
                 "inventory_number", "catalog_number",
                 "location", "exhibited_at", "notes"]

    with open(OUT_READ, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=read_cols, extrasaction="ignore")
        writer.writeheader()
        for rec in records:
            mats     = rec.get("materials") or []
            mat_qids = rec.get("materials_qids") or []
            writer.writerow({
                "title":           rec.get("title") or "",
                "creator":         rec.get("creator") or "John Baldessari",
                "creator_qid":     rec.get("creator_qid") or "Q312793",
                "date":            rec.get("date") or "",
                "object_type":     rec.get("object_type") or "",
                "object_type_qid": rec.get("object_type_qid") or "",
                "materials":       "; ".join(str(m) for m in mats),
                "materials_qids":  "; ".join(str(q) for q in mat_qids if q),
                "height_cm":       rec.get("height_cm") if rec.get("height_cm") is not None else "",
                "width_cm":        rec.get("width_cm")  if rec.get("width_cm")  is not None else "",
                "depth_cm":        rec.get("depth_cm")  if rec.get("depth_cm")  is not None else "",
                "inventory_number": rec.get("inventory_number") or "",
                "catalog_number":   rec.get("catalog_number")   or "",
                "location":        "Sprengel Museum Hannover",
                "exhibited_at":    "Q673907",
                "notes":           "",
            })
    print(f"  Wrote {OUT_READ} ({OUT_READ.stat().st_size:,} bytes, {len(records)} rows)")


step6_write_csv(resolved)

# ── Final report ─────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(f"OCR validity ratio:         {validity_ratio:.4f} ({'PASS' if validity_ratio >= 0.80 else 'FAIL'})")
print(f"Chunks processed:           {num_chunks}")
print(f"Raw records (before dedup): {raw_count}")
print(f"Records after dedup:        {len(resolved)}")
for f in [OUT_CSV, OUT_READ, FAIL_LOG]:
    if f.exists():
        print(f"{f.name}: {f.stat().st_size:,} bytes")
print("=" * 60)